##Setup

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import optuna
from optuna.samplers import TPESampler
import joblib
from scipy.stats import loguniform, uniform, randint

from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, average_precision_score, balanced_accuracy_score, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

INNER_FOLDS   = 5
INNER_REPEATS = 3
N_TRIALS      = 200      

SELECT_SCORE_FUNC = f_classif  

FONT_TITLE  = 22
FONT_AXIS   = 18
FONT_LEGEND = 16
FONT_TICK   = 16

plt.rcParams.update({
    "font.size":        FONT_AXIS,
    "axes.titlesize":   FONT_TITLE,
    "axes.labelsize":   FONT_AXIS,
    "xtick.labelsize":  FONT_TICK,
    "ytick.labelsize":  FONT_TICK,
    "legend.fontsize":  FONT_LEGEND,
    "figure.titlesize": FONT_TITLE,
})

RESULTS_ROOT = "resultados_8modelos"
METHODS      = ["optuna", "random_search", "grid_search"]
METHOD_LABELS = {"optuna": "Optuna","random_search": "Random Search","grid_search":   "Grid Search",}

METHOD_DIRS = {}
for _method in METHODS:
    _plots_dir  = os.path.join(RESULTS_ROOT, _method, "plots")
    _models_dir = os.path.join(RESULTS_ROOT, _method, "models")
    os.makedirs(_plots_dir,  exist_ok=True)
    os.makedirs(_models_dir, exist_ok=True)
    METHOD_DIRS[_method] = {"plots": _plots_dir, "models": _models_dir}

COMPARISON_DIR = os.path.join(RESULTS_ROOT, "comparacion")
os.makedirs(COMPARISON_DIR, exist_ok=True)

ALL_RESULTS = {method: {} for method in METHODS}

print("Estructura de carpetas lista:")
for _method, _dirs in METHOD_DIRS.items():
    print(f"  {_method:15s} -> {_dirs['plots']}  |  {_dirs['models']}")
print(f"  {'comparacion':15s} -> {COMPARISON_DIR}")

Carga y preparación de datos

In [ ]:
df = pd.read_csv(r"C:\Users\Sergi\Desktop\Investigacion de materiales\Codigo definitivo\bdelectrolitos.csv")
df = df.drop_duplicates().sample(frac=1, random_state=22).reset_index(drop=True)

UMBRAL = 99
df["desempeno"] = (df["CE"] >= UMBRAL).astype(int)

cols_drop = ["Electrolyte", "CE", "LCE", "desempeno"]
X = df.drop(columns=[c for c in cols_drop if c in df.columns])
y = df["desempeno"]

print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")
print(f"Distribucion:\n{y.value_counts()}")
print(f"Positivos: {y.mean():.2%}")

Split hold-out (TEST intocable)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"TRAIN: {X_train.shape[0]} muestras")
print(f"TEST : {X_test.shape[0]} muestras  <- intocable hasta la evaluacion final")

In [ ]:
import joblib
import os

SPLIT_DIR = os.path.join(RESULTS_ROOT, "particion_datos")
os.makedirs(SPLIT_DIR, exist_ok=True)

joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, os.path.join(SPLIT_DIR, "train_test_split.pkl"))

print(f"Particion guardada en: {os.path.join(SPLIT_DIR, 'train_test_split.pkl')}")

Rango de `k` para SelectKBest (según el número de columnas polinómicas)

In [ ]:
_poly_probe = PolynomialFeatures(degree=2, include_bias=False)
N_FEATURES_POLY = _poly_probe.fit_transform(X_train).shape[1]

K_MIN = 10
K_MAX = min(20, N_FEATURES_POLY)

print(f"Features originales    : {X_train.shape[1]}")
print(f"Features polinomicas    : {N_FEATURES_POLY}")
print(f"Rango de seleccion (k)  : [{K_MIN}, {K_MAX}]")

Funciones auxiliares: búsqueda de hiperparámetros + evaluación

In [ ]:
def _prefix_params(d):
    """select_k -> select__k ; cualquier otro -> model__<param>"""
    out = {}
    for k, v in d.items():
        if k == "select_k":
            out["select__k"] = v
        else:
            out[f"model__{k}"] = v
    return out


def _strip_prefix(d):
    out = {}
    for k, v in d.items():
        if k.startswith("select__"):
            out["select_k"] = v
        elif k.startswith("model__"):
            out[k[len("model__"):]] = v
        else:
            out[k] = v
    return out


def _search_hyperparams(X_tr, y_tr, inner_cv, ratio, make_pipeline_fn,search_method, n_evals, seed,make_objective_fn=None,param_grid=None, param_distributions=None):

    t0 = time.time()

    if search_method == "optuna":
        objective = make_objective_fn(X_tr, y_tr, inner_cv, ratio)
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_evals, show_progress_bar=False)
        best_params = study.best_params
        n_real = len(study.trials)

    elif search_method in ("random_search", "grid_search"):
        base_pipeline = make_pipeline_fn({}, ratio)

        if search_method == "random_search":
            if isinstance(param_distributions, list):
                space = [_prefix_params(d) for d in param_distributions]
            else:
                space = _prefix_params(param_distributions)
            searcher = RandomizedSearchCV(
                base_pipeline, param_distributions=space,
                n_iter=n_evals, scoring="roc_auc", cv=inner_cv,
                random_state=seed, n_jobs=-1, refit=False,
            )
        else:  # grid_search
            if isinstance(param_grid, list):
                space = [_prefix_params(d) for d in param_grid]
            else:
                space = _prefix_params(param_grid)
            searcher = GridSearchCV(
                base_pipeline, param_grid=space,
                scoring="roc_auc", cv=inner_cv, n_jobs=-1, refit=False,
            )

        searcher.fit(X_tr, y_tr)
        best_params = _strip_prefix(searcher.best_params_)
        n_real = len(searcher.cv_results_["params"])

    else:
        raise ValueError(f"search_method desconocido: {search_method}")

    elapsed = time.time() - t0
    return best_params, n_real, elapsed


def bootstrap_metric_ci(y_true, y_prob, metric_fn=roc_auc_score, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue  # evita folds bootstrap sin ambas clases
        scores.append(metric_fn(y_true[idx], y_prob[idx]))
    ci_low, ci_med, ci_high = np.percentile(scores, [2.5, 50, 97.5])
    return round(ci_low, 4), round(ci_med, 4), round(ci_high, 4)


def _ensure_predict_proba(pipeline):
    """LinearSVC y RidgeClassifier no tienen predict_proba nativo.
    Se envuelve el paso final en CalibratedClassifierCV para poder
    calcular probabilidades (curvas ROC, bootstrap CI, etc.)."""
    last_name, last_step = pipeline.steps[-1]
    if hasattr(last_step, "predict_proba"):
        return pipeline
    calibrated = CalibratedClassifierCV(last_step, method="sigmoid", cv=5)
    new_steps = pipeline.steps[:-1] + [(last_name, calibrated)]
    return Pipeline(new_steps)


def run_model(name, make_pipeline_fn, search_method="optuna",make_objective_fn=None,param_distributions=None,param_grid=None,n_trials=N_TRIALS):

    plots_dir  = METHOD_DIRS[search_method]["plots"]
    models_dir = METHOD_DIRS[search_method]["models"]
    label      = METHOD_LABELS[search_method]

    print(f"\n{name} ({label})")

    ratio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]
    inner_cv = RepeatedStratifiedKFold(n_splits=INNER_FOLDS, n_repeats=INNER_REPEATS, random_state=42    )

    print(f"  Busqueda de hiperparametros en TRAIN ({label}):")
    best_params, n_evals, elapsed = _search_hyperparams(X_train, y_train, inner_cv, ratio, make_pipeline_fn,search_method, n_trials, seed=42,
        make_objective_fn=make_objective_fn, param_grid=param_grid, param_distributions=param_distributions,)
    print(f"  Mejores params: {best_params}")
    print(f"  Evaluaciones: {n_evals} | Tiempo busqueda: {elapsed:.1f}s")

    final_pipeline = make_pipeline_fn(best_params, ratio)
    final_pipeline = _ensure_predict_proba(final_pipeline)
    final_pipeline.fit(X_train, y_train)

    probs_train = final_pipeline.predict_proba(X_train)[:, 1]
    preds_train = final_pipeline.predict(X_train)
    probs_test  = final_pipeline.predict_proba(X_test)[:, 1]
    preds_test  = final_pipeline.predict(X_test)

    auc_ci_low, auc_ci_med, auc_ci_high = bootstrap_metric_ci(
        y_test, probs_test, metric_fn=roc_auc_score, n_boot=1000, seed=42
    )

    metrics = {
        "modelo"                 : name,
        "metodo_busqueda"        : label,
        "train_auc"              : round(roc_auc_score(y_train, probs_train), 4),
        "train_acc"              : round(accuracy_score(y_train, preds_train), 4),
        "train_f1"               : round(f1_score(y_train, preds_train), 4),
        "test_auc"               : round(roc_auc_score(y_test, probs_test), 4),
        "test_auc_ci_low"        : auc_ci_low,
        "test_auc_ci_median"     : auc_ci_med,
        "test_auc_ci_high"       : auc_ci_high,
        "test_acc"               : round(accuracy_score(y_test, preds_test), 4),
        "test_f1"                : round(f1_score(y_test, preds_test), 4),
        "test_pr_auc"            : round(average_precision_score(y_test, probs_test), 4),
        "test_balanced_accuracy" : round(balanced_accuracy_score(y_test, preds_test), 4),
        "n_evaluaciones"         : n_evals,
        "tiempo_busqueda_seg"    : round(elapsed, 2),
        "best_params"            : str(best_params),
    }
    metrics["gap_auc"] = round(metrics["train_auc"] - metrics["test_auc"], 4)

    print(f"\n  ENTRENAMIENTO (TRAIN)  AUC:{metrics['train_auc']}  ACC:{metrics['train_acc']}  F1:{metrics['train_f1']}")
    print(f"  PRUEBA (TEST)          AUC:{metrics['test_auc']}  ACC:{metrics['test_acc']}  F1:{metrics['test_f1']}  "
          f"PR-AUC:{metrics['test_pr_auc']}  Bal.Acc:{metrics['test_balanced_accuracy']}")
    print(f"  TEST AUC bootstrap (IC 95%): {metrics['test_auc_ci_low']} - {metrics['test_auc_ci_high']}  "
          f"(mediana: {metrics['test_auc_ci_median']})")
    print(f"  Diferencia (Gap) AUC: {metrics['gap_auc']}")

    print("\n  Reporte train:")
    print(classification_report(y_train, preds_train, target_names=["Clase 0", "Clase 1"]))
    print("\n  Reporte test:")
    print(classification_report(y_test, preds_test, target_names=["Clase 0", "Clase 1"]))
    print()

    safe_name = name.replace(" ", "_")

    # ROC (TRAIN + TEST)
    fig, ax = plt.subplots(figsize=(6, 5))
    for split_label, y_true, y_prob, ls in [
        ("Train", y_train, probs_train, "--"),
        ("Test",  y_test,  probs_test,  "-"),
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, linestyle=ls, linewidth=2,
                label=f"{split_label} (AUC={auc_val:.4f})")
    ax.plot([0, 1], [0, 1], "k:", linewidth=1)
    ax.set_xlabel("False Positive Rate", fontsize=FONT_AXIS)
    ax.set_ylabel("True Positive Rate", fontsize=FONT_AXIS)
    ax.set_title(f"ROC Curve — {name} ({label})", fontsize=FONT_TITLE, fontweight="bold")
    ax.legend(fontsize=FONT_LEGEND)
    ax.grid(True, alpha=0.3)
    roc_path = os.path.join(plots_dir, f"{safe_name}_roc.png")
    fig.savefig(roc_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  ROC guardada: {roc_path}")

    # Matriz de confusion (TEST)
    cm = confusion_matrix(y_test, preds_test)
    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    for text in ax.texts:
        text.set_fontsize(FONT_AXIS)
    ax.set_title(f"Confusion Matrix — {name} ({label})", fontsize=FONT_TITLE, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=FONT_AXIS)
    ax.set_ylabel("Real", fontsize=FONT_AXIS)
    cm_path = os.path.join(plots_dir, f"{safe_name}_confusion.png")
    fig.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Matriz de confusión guardada: {cm_path}")

    model_path = os.path.join(models_dir, f"{safe_name}.pkl")
    joblib.dump(final_pipeline, model_path)
    print(f"  Modelo guardado: {model_path}")

    ALL_RESULTS[search_method][name] = {
        "metrics"    : metrics,
        "model"      : final_pipeline,
        "probs_test" : probs_test,
        "preds_test" : preds_test,
    }

    return final_pipeline, metrics

Regresión Logística

In [ ]:
def lr_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params,
         "class_weight": {0: 1, 1: ratio},
         "solver": "saga",
         "max_iter": 10000,
         "random_state": 42}
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  LogisticRegression(**p)),
    ])


def lr_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])
        params = {
            "select_k": trial.suggest_int("select_k", K_MIN, K_MAX),
            "penalty":  penalty,
            "C":        trial.suggest_float("C", 2, 1e3, log=True),
        }
        if penalty == "elasticnet":
            params["l1_ratio"] = trial.suggest_float("l1_ratio", 0.0, 1.0)
        pipe = lr_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            probs = pipe.predict_proba(X_tr.iloc[ival])[:, 1]
            scores.append(roc_auc_score(y_tr.iloc[ival], probs))
        return np.mean(scores)
    return objective


lr_param_distributions = [
    {"select_k": randint(K_MIN, K_MAX + 1), "penalty": ["l1", "l2"], "C": loguniform(2, 1e3)},
    {"select_k": randint(K_MIN, K_MAX + 1), "penalty": ["elasticnet"], "C": loguniform(2, 1e3),
     "l1_ratio": uniform(0, 1)},
]

lr_grid = [
    {"select_k": [30, 60, 90], "penalty": ["l1", "l2"], "C": [5, 10, 50, 200]},
    {"select_k": [30, 60, 90], "penalty": ["elasticnet"], "C": [5, 10, 50, 200], "l1_ratio": [0.2, 0.5, 0.8]},
]

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=lr_make_objective)),
    ("random_search",  dict(param_distributions=lr_param_distributions)),
    ("grid_search",   dict(param_grid=lr_grid)),
]:
    run_model(name="Regresion Logistica", make_pipeline_fn=lr_make_pipeline,
              search_method=_method, **_kwargs)

LinearSVC

In [ ]:
def svc_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params,
         "class_weight": {0: 1, 1: ratio},
         "max_iter": 10000,
         "dual": "auto",
         "random_state": 42}
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  LinearSVC(**p)),
    ])


def svc_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {"select_k": trial.suggest_int("select_k", K_MIN, K_MAX),"C": trial.suggest_float("C", 1e-3, 1e2, log=True), }
        pipe = svc_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            dec = pipe.decision_function(X_tr.iloc[ival])
            scores.append(roc_auc_score(y_tr.iloc[ival], dec))
        return np.mean(scores)
    return objective


svc_param_distributions = { "select_k": randint(K_MIN, K_MAX + 1),  "C": loguniform(1e-3, 1e2),}

svc_grid = {  "select_k": [30, 60, 90],    "C":        [0.01, 0.1, 1, 10, 100],}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=svc_make_objective)),
    ("random_search",  dict(param_distributions=svc_param_distributions)),
    ("grid_search",   dict(param_grid=svc_grid)),
]:
    run_model(name="LinearSVC", make_pipeline_fn=svc_make_pipeline,
              search_method=_method, **_kwargs)

RidgeClassifier

In [ ]:
def ridge_make_pipeline(params, ratio): select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params,
         "class_weight": {0: 1, 1: ratio},
         "random_state": 42}
    return Pipeline([("poly",   PolynomialFeatures(degree=2, include_bias=False)), ("scaler", StandardScaler()), ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  RidgeClassifier(**p)),
    ])


def ridge_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {"select_k": trial.suggest_int("select_k", K_MIN, K_MAX),"alpha":    trial.suggest_float("alpha", 1e-3, 1e2, log=True),  }
        pipe = ridge_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            dec = pipe.decision_function(X_tr.iloc[ival])
            scores.append(roc_auc_score(y_tr.iloc[ival], dec))
        return np.mean(scores)
    return objective


ridge_param_distributions = {
    "select_k": randint(K_MIN, K_MAX + 1),
    "alpha":    loguniform(1e-3, 1e2),
}

ridge_grid = {
    "select_k": [30, 60, 90],
    "alpha":    [0.01, 0.1, 1, 10, 100],
}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=ridge_make_objective)),
    ("random_search",  dict(param_distributions=ridge_param_distributions)),
    ("grid_search",   dict(param_grid=ridge_grid)),
]:
    run_model(name="RidgeClassifier", make_pipeline_fn=ridge_make_pipeline,
              search_method=_method, **_kwargs)

XGBoost

In [ ]:
def xgb_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params,
         "scale_pos_weight": ratio,
         "eval_metric": "logloss",
         "random_state": 42,
         "n_jobs": -1,
         "verbosity": 0}
    return Pipeline([("poly",   PolynomialFeatures(degree=2, include_bias=False)), ("scaler", StandardScaler()), ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  XGBClassifier(**p)),
    ])


def xgb_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {
            "select_k":         trial.suggest_int("select_k", K_MIN, K_MAX),
            "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
            "max_depth":        trial.suggest_int("max_depth", 3, 15),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1),
            "subsample":        trial.suggest_float("subsample", 0.8, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
            "gamma":            trial.suggest_float("gamma", 0, 2),
        }
        pipe = xgb_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            probs = pipe.predict_proba(X_tr.iloc[ival])[:, 1]
            scores.append(roc_auc_score(y_tr.iloc[ival], probs))
        return np.mean(scores)
    return objective


xgb_param_distributions = {
    "select_k":         randint(K_MIN, K_MAX + 1),
    "n_estimators":     randint(100, 1001),
    "max_depth":        randint(3, 13),
    "learning_rate":    loguniform(0.01, 0.2),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "gamma":            uniform(0, 3),
}

xgb_grid = {
    "select_k":      [30, 60, 90],
    "n_estimators":  [300, 600, 900],
    "max_depth":     [4, 6, 9],
    "learning_rate": [0.02, 0.06, 0.15],
    "gamma":         [0, 2],
}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=xgb_make_objective)),
    ("random_search",  dict(param_distributions=xgb_param_distributions)),
    ("grid_search",   dict(param_grid=xgb_grid)),
]:
    run_model(name="XGBoost", make_pipeline_fn=xgb_make_pipeline,
              search_method=_method, **_kwargs)

LightGBM

In [ ]:
def lgbm_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params, "scale_pos_weight": ratio,"random_state": 42, "n_jobs": -1, "verbosity": -1}
    
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  LGBMClassifier(**p)),
    ])


def lgbm_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {
            "select_k":         trial.suggest_int("select_k", K_MIN, K_MAX),
            "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
            "max_depth":        trial.suggest_int("max_depth", 3, 15),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1),
            "num_leaves":       trial.suggest_int("num_leaves", 15, 127),
            "subsample":        trial.suggest_float("subsample", 0.8, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
        }
        pipe = lgbm_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            probs = pipe.predict_proba(X_tr.iloc[ival])[:, 1]
            scores.append(roc_auc_score(y_tr.iloc[ival], probs))
        return np.mean(scores)
    return objective


lgbm_param_distributions = {
    "select_k":         randint(K_MIN, K_MAX + 1),
    "n_estimators":     randint(100, 1001),
    "max_depth":        randint(3, 13),
    "learning_rate":    loguniform(0.01, 0.2),
    "num_leaves":       randint(15, 128),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
}

lgbm_grid = {
    "select_k":      [30, 60, 90],
    "n_estimators":  [300, 600, 900],
    "max_depth":     [4, 6, 9],
    "learning_rate": [0.02, 0.06, 0.15],
    "num_leaves":    [31, 63],
}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=lgbm_make_objective)),
    ("random_search",  dict(param_distributions=lgbm_param_distributions)),
    ("grid_search",   dict(param_grid=lgbm_grid)),
]:
    run_model(name="LightGBM", make_pipeline_fn=lgbm_make_pipeline,
              search_method=_method, **_kwargs)

Random Forest

In [ ]:
def rf_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    p = {**model_params,
         "class_weight": {0: 1, 1: ratio},
         "random_state": 42,
         "n_jobs": -1}
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  RandomForestClassifier(**p)),
    ])


def rf_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {
            "select_k":          trial.suggest_int("select_k", K_MIN, K_MAX),
            "n_estimators":      trial.suggest_int("n_estimators", 100, 800),
            "max_depth":         trial.suggest_int("max_depth", 3, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
        pipe = rf_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            probs = pipe.predict_proba(X_tr.iloc[ival])[:, 1]
            scores.append(roc_auc_score(y_tr.iloc[ival], probs))
        return np.mean(scores)
    return objective


rf_param_distributions = {
    "select_k":          randint(K_MIN, K_MAX + 1),
    "n_estimators":      randint(100, 801),
    "max_depth":         randint(3, 21),
    "min_samples_split": randint(2, 21),
    "min_samples_leaf":  randint(1, 11),
    "max_features":      ["sqrt", "log2"],
}

rf_grid = {
    "select_k":          [30, 60, 90],
    "n_estimators":      [200, 400, 700],
    "max_depth":         [5, 10, 20],
    "min_samples_split": [2, 6, 12],
    "max_features":      ["sqrt", "log2"],
}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=rf_make_objective)),
    ("random_search",  dict(param_distributions=rf_param_distributions)),
    ("grid_search",   dict(param_grid=rf_grid)),
]:
    run_model(name="Random Forest", make_pipeline_fn=rf_make_pipeline,
              search_method=_method, **_kwargs)

BaggingClassifier

In [ ]:
def bag_make_pipeline(params, ratio):
    select_k = params.get("select_k", min(50, K_MAX))
    model_params = {k: v for k, v in params.items() if k != "select_k"}
    base = DecisionTreeClassifier(class_weight={0: 1, 1: ratio}, random_state=42)
    p = {**model_params,
         "estimator": base,
         "random_state": 42,
         "n_jobs": -1}
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=SELECT_SCORE_FUNC, k=select_k)),
        ("model",  BaggingClassifier(**p)),
    ])


def bag_make_objective(X_tr, y_tr, inner_cv, ratio):
    def objective(trial):
        params = {
            "select_k":     trial.suggest_int("select_k", K_MIN, K_MAX),
            "n_estimators": trial.suggest_int("n_estimators", 50, 500),
            "max_samples":  trial.suggest_float("max_samples", 0.5, 1.0),
            "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        }
        pipe = bag_make_pipeline(params, ratio)
        scores = []
        for itr, ival in inner_cv.split(X_tr, y_tr):
            pipe.fit(X_tr.iloc[itr], y_tr.iloc[itr])
            probs = pipe.predict_proba(X_tr.iloc[ival])[:, 1]
            scores.append(roc_auc_score(y_tr.iloc[ival], probs))
        return np.mean(scores)
    return objective


bag_param_distributions = {
    "select_k":     randint(K_MIN, K_MAX + 1),
    "n_estimators": randint(50, 501),
    "max_samples":  uniform(0.5, 0.5),
    "max_features": uniform(0.5, 0.5),
}

bag_grid = {
    "select_k":     [30, 60, 90],
    "n_estimators": [100, 200, 400],
    "max_samples":  [0.6, 0.8, 1.0],
    "max_features": [0.6, 0.8, 1.0],
}

for _method, _kwargs in [
    ("optuna",        dict(make_objective_fn=bag_make_objective)),
    ("random_search",  dict(param_distributions=bag_param_distributions)),
    ("grid_search",   dict(param_grid=bag_grid)),
]:
    run_model(name="Bagging", make_pipeline_fn=bag_make_pipeline,
              search_method=_method, **_kwargs)

Tabla comparativa consolidada (CSV único con las 24 corridas)

Una fila por combinación modelo × método de búsqueda (8 modelos × 3 métodos = 24 filas).

In [ ]:
rows = []
for method, models_dict in ALL_RESULTS.items():
    for model_name, result in models_dict.items():
        rows.append(result["metrics"])

df_comparacion = pd.DataFrame(rows)

orden_cols = [
    "modelo", "metodo_busqueda",
    "train_auc", "test_auc", "test_auc_ci_low", "test_auc_ci_median", "test_auc_ci_high",
    "test_acc", "test_f1", "test_pr_auc", "test_balanced_accuracy", "gap_auc",
    "n_evaluaciones", "tiempo_busqueda_seg",
    "best_params",
]
df_comparacion = df_comparacion[orden_cols].sort_values(["modelo", "metodo_busqueda"]).reset_index(drop=True)

csv_path = os.path.join(COMPARISON_DIR, "resultados_todos.csv")
df_comparacion.to_csv(csv_path, index=False)
print(f"CSV consolidado guardado en: {csv_path}")
print(f"Filas: {len(df_comparacion)}  (esperado: {len(METHODS)} metodos x 8 modelos = {len(METHODS) * 8})")

df_comparacion

Gráficas comparativas entre métodos de búsqueda

In [ ]:
fig, axes = plt.subplots(1, len(METHODS), figsize=(6 * len(METHODS), 5.5), sharey=True)
colors = ["#e63946", "#2a9d8f", "#e9c46a", "#457b9d", "#8338ec", "#ff006e", "#3a86ff", "#06d6a0"]

for ax, method in zip(axes, METHODS):
    for (model_name, result), color in zip(ALL_RESULTS[method].items(), colors):
        fpr, tpr, _ = roc_curve(y_test, result["probs_test"])
        auc_val = result["metrics"]["test_auc"]
        ax.plot(fpr, tpr, color=color, linewidth=2.0,
                label=f"{model_name} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k:", linewidth=1)
    ax.set_title(METHOD_LABELS[method], fontsize=FONT_TITLE, fontweight="bold")
    ax.set_xlabel("False Positive Rate", fontsize=FONT_AXIS)
    ax.legend(fontsize=FONT_LEGEND - 4, loc="lower right")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("True Positive Rate", fontsize=FONT_AXIS)
fig.suptitle("Test ROC Curves by Search Method", fontsize=FONT_TITLE, fontweight="bold")
fig.tight_layout()

comp_roc_path = os.path.join(COMPARISON_DIR, "comparacion_roc_por_metodo.png")
fig.savefig(comp_roc_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Grafica guardada: {comp_roc_path}")

In [ ]:
modelos = sorted({m for d in ALL_RESULTS.values() for m in d})
x = np.arange(len(modelos))
width = 0.25
method_colors = {"optuna": "#457b9d", "random_search": "#e9c46a", "grid_search": "#e63946"}

# grafica 1: test auc por modelo y metodo
fig, ax = plt.subplots(figsize=(12, 5.5))
for i, method in enumerate(METHODS):
    vals = [ALL_RESULTS[method][m]["metrics"]["test_auc"] if m in ALL_RESULTS[method] else np.nan
            for m in modelos]
    ax.bar(x + (i - 1) * width, vals, width, label=METHOD_LABELS[method], color=method_colors[method])

ax.set_xticks(x)
ax.set_xticklabels(modelos, rotation=20, ha="right", fontsize=FONT_TICK)
ax.set_ylabel("Test AUC", fontsize=FONT_AXIS)
ax.set_title("Search Method Comparison — Test AUC", fontsize=FONT_TITLE, fontweight="bold")
ax.legend(fontsize=FONT_LEGEND)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

path_auc = os.path.join(COMPARISON_DIR, "comparacion_test_auc.png")
fig.savefig(path_auc, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Grafica guardada: {path_auc}")

# grafica 2: tiempo de busqueda por modelo y metodo
fig, ax = plt.subplots(figsize=(12, 5.5))
for i, method in enumerate(METHODS):
    tiempos = [ALL_RESULTS[method][m]["metrics"]["tiempo_busqueda_seg"] if m in ALL_RESULTS[method] else np.nan
               for m in modelos]
    ax.bar(x + (i - 1) * width, tiempos, width, label=METHOD_LABELS[method], color=method_colors[method])

ax.set_xticks(x)
ax.set_xticklabels(modelos, rotation=20, ha="right", fontsize=FONT_TICK)
ax.set_ylabel("Search Time (seconds)", fontsize=FONT_AXIS)
ax.set_title("Search Method Comparison — Compute Cost", fontsize=FONT_TITLE, fontweight="bold")
ax.legend(fontsize=FONT_LEGEND)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

path_time = os.path.join(COMPARISON_DIR, "comparacion_tiempo_busqueda.png")
fig.savefig(path_time, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Grafica guardada: {path_time}")